# Crossref Metadata Enrichment

Queries the [Crossref API](https://www.crossref.org/) to enrich article metadata extracted from XML with DOIs, citation counts, journal names, references, and subject classifications. Uses a multi-strategy search (title similarity + author fallback) and a configurable similarity threshold to accept matches.

**Input:** `output/metadata_from_xml.csv` — produced by `ExtractXML.ipynb`  
**Output:**
- `output/crossref_experiment_FULL.csv` — top-5 Crossref candidates per article (for threshold tuning)
- `output/metadata_enriched_FINAL.csv` — final enriched dataset after applying the similarity threshold
- `output/crossref_CHECK.csv` — simplified view for manual validation

In [ ]:
import json
import re
import time
from difflib import SequenceMatcher
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
from habanero import Crossref
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path("..").resolve()
OUTPUT_DIR   = PROJECT_ROOT / "output"

INPUT_CSV      = OUTPUT_DIR / "metadata_from_xml.csv"
EXPERIMENT_CSV = OUTPUT_DIR / "crossref_experiment_FULL.csv"
ENRICHED_CSV   = OUTPUT_DIR / "metadata_enriched_FINAL.csv"
CHECK_CSV      = OUTPUT_DIR / "crossref_CHECK.csv"

SIMILARITY_THRESHOLD = 0.90  # minimum title similarity to accept a Crossref match
MAX_CANDIDATES       = 5     # Crossref results stored per article (for inspection)
RATE_LIMIT_EVERY     = 8     # pause 1 s after every N requests to stay within 10 req/s

In [ ]:
cr = Crossref()

In [ ]:
def extract_authors(item: dict) -> Optional[str]:
    """Return a semicolon-separated author string from a Crossref item, or None."""
    authors = item.get("author")
    if not authors or not isinstance(authors, list):
        return None

    parts = []
    for a in authors[:10]:
        if not isinstance(a, dict):
            continue
        family = a.get("family", "")
        given  = a.get("given", "")
        if family and given:
            parts.append(f"{given} {family}")
        elif family:
            parts.append(family)

    return "; ".join(parts) if parts else None


def extract_year(item: dict) -> Optional[int]:
    """Return the publication year from a Crossref item, or None."""
    for key in ("published-print", "published-online", "published"):
        date_parts = item.get(key, {}).get("date-parts")
        if date_parts and date_parts[0]:
            return date_parts[0][0]
    return None


def calculate_title_similarity(title1: Optional[str], title2: Optional[str]) -> float:
    """Return a 0–1 similarity score between two title strings (case-insensitive)."""
    if not title1 or not title2:
        return 0.0
    return SequenceMatcher(None, title1.lower().strip(), title2.lower().strip()).ratio()


def extract_metadata(item: dict) -> dict:
    """Extract all available fields from a Crossref API item into a flat dict."""
    references = [
        {
            "doi":        ref.get("DOI"),
            "title":      ref.get("article-title"),
            "year":       ref.get("year"),
            "author":     ref.get("author"),
            "journal":    ref.get("journal-title"),
            "volume":     ref.get("volume"),
            "first_page": ref.get("first-page"),
        }
        for ref in item.get("reference", [])
        if isinstance(ref, dict)
    ]

    event = item.get("event", {})
    issn_list = item.get("ISSN", [])
    isbn_list = item.get("ISBN", [])

    return {
        "doi":                      item.get("DOI"),
        "crossref_title":           (item.get("title") or [""])[0] or None,
        "crossref_authors":         extract_authors(item),
        "crossref_year":            extract_year(item),
        "crossref_journal":         (item.get("container-title") or [None])[0],
        "crossref_type":            item.get("type"),
        "crossref_publisher":       item.get("publisher"),
        "crossref_url":             item.get("URL"),
        "crossref_abstract":        item.get("abstract"),
        "crossref_references_count":item.get("reference-count", 0),
        "crossref_cited_by_count":  item.get("is-referenced-by-count", 0),
        "crossref_score":           item.get("score"),
        "crossref_references":      json.dumps(references),
        "crossref_subject":         json.dumps(item.get("subject", [])),
        "crossref_issn":            issn_list[0] if issn_list else None,
        "crossref_isbn":            isbn_list[0] if isbn_list else None,
        "crossref_volume":          item.get("volume"),
        "crossref_issue":           item.get("issue"),
        "crossref_page":            item.get("page"),
        "crossref_article_number":  item.get("article-number"),
        "crossref_event_name":      event.get("name"),
        "crossref_event_location":  event.get("location"),
        "crossref_language":        item.get("language"),
    }

In [ ]:
def search_crossref(title: Optional[str] = None, author: Optional[str] = None,
                    max_results: int = MAX_CANDIDATES) -> List[dict]:
    """Query the Crossref API and return a list of raw item dicts."""
    try:
        params: dict = {"limit": max_results}
        if title:  
            params["query_title"] = title
        if author: 
            params["query_author"] = author

        results = cr.works(**params)
        if not results or "message" not in results:
            return []
        return results["message"].get("items", [])
    except Exception as e:
        print(f"Crossref API error: {e}")
        return []


def search_crossref_multi_strategy(row: pd.Series) -> List[dict]:
    """
    Search Crossref using a two-stage strategy:
      1. If a valid title exists — search by title (+ first author surname).
      2. If no title — fall back to author-only search.

    Returns a list of enriched metadata dicts (up to MAX_CANDIDATES).
    """
    title   = row.get("title")
    authors = row.get("authors")

    # Title is valid if it has enough alphabetic content
    title_str, has_valid_title = "", False
    if title and str(title).strip():
        title_str = str(title).strip()
        alpha = len(re.findall(r"[a-zA-Z]", title_str))
        total = len(title_str.replace(" ", ""))
        has_valid_title = total > 10 and alpha / total >= 0.5

    # Extract first author surname
    surname = None
    if authors and str(authors).strip():
        first_author = str(authors).split(";")[0].strip()
        candidate = first_author.split()[-1] if first_author else ""
        surname = candidate if len(candidate) >= 3 else None

    if has_valid_title:
        items = search_crossref(title=title_str, author=surname)
        if items:
            results = []
            for item in items:
                meta = extract_metadata(item)
                meta["title_similarity"] = calculate_title_similarity(title_str, meta.get("crossref_title"))
                meta["search_strategy"]  = "with_title"
                results.append(meta)
            return results

    if surname:
        items = search_crossref(author=surname)
        if items:
            results = []
            for item in items:
                meta = extract_metadata(item)
                meta["title_similarity"] = None
                meta["search_strategy"]  = "author_only"
                results.append(meta)
            return results

    return []

In [ ]:
def enrich_dataset_with_crossref(input_csv: Path, output_csv: Path,
                                  sample_size: Optional[int] = None) -> pd.DataFrame:
    """
    Query Crossref for every article in input_csv and save up to MAX_CANDIDATES
    candidates per article to output_csv for later threshold tuning.

    Args:
        input_csv:   CSV produced by ExtractXML.ipynb.
        output_csv:  Destination for the experiment results.
        sample_size: If set, only process the first N rows (useful for testing).

    Returns:
        DataFrame with original metadata + up to 5 Crossref candidates per row.
    """
    df = pd.read_csv(input_csv)
    if sample_size:
        print(f"Test mode: processing first {sample_size} articles")
        df = df.head(sample_size)

    print(f"Loaded {len(df)} articles from {input_csv}")

    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        results = search_crossref_multi_strategy(row)

        exp_row = {
            "idx":              idx,
            "pdf_file":         row.get("pdf_file", ""),
            "original_title":   row.get("title", ""),
            "original_authors": row.get("authors", ""),
            "original_year":    row.get("year", ""),
            "search_strategy":  results[0]["search_strategy"] if results else "not_found",
            "num_results":      len(results),
        }

        for i, result in enumerate(results[:MAX_CANDIDATES], 1):
            for field in ("crossref_title", "crossref_authors", "crossref_year", "doi",
                          "crossref_score", "crossref_references", "crossref_subject",
                          "crossref_journal", "crossref_publisher", "crossref_type",
                          "crossref_abstract", "crossref_issn", "crossref_volume",
                          "crossref_issue", "crossref_page", "crossref_event_name",
                          "crossref_language"):
                exp_row[f"{field}_{i}"] = result.get(field, "")
            exp_row[f"similarity_{i}"] = result.get("title_similarity")

        rows.append(exp_row)

        if (idx + 1) % RATE_LIMIT_EVERY == 0:
            time.sleep(1)

    df_out = pd.DataFrame(rows)
    df_out.to_csv(output_csv, index=False, encoding="utf-8")

    print(f"\nSaved {len(df_out)} rows -> {output_csv}")
    print(f"  Matched by title:  {(df_out['search_strategy'] == 'with_title').sum()}")
    print(f"  Matched by author: {(df_out['search_strategy'] == 'author_only').sum()}")
    print(f"  Not found:         {(df_out['search_strategy'] == 'not_found').sum()}")
    return df_out


def apply_threshold_and_enrich(experiment_csv: Path, output_csv: Path,
                                threshold: float = SIMILARITY_THRESHOLD) -> pd.DataFrame:
    """
    Apply a similarity threshold to the experiment CSV and produce the final enriched dataset.
    Articles matched by title are accepted only if similarity >= threshold.
    Articles matched by author alone take the first Crossref result without a similarity check.

    Args:
        experiment_csv: Output of enrich_dataset_with_crossref().
        output_csv:     Destination for the final enriched CSV.
        threshold:      Minimum title similarity to accept a match (default: SIMILARITY_THRESHOLD).

    Returns:
        Final enriched DataFrame.
    """
    df_exp = pd.read_csv(experiment_csv)
    print(f"Applying threshold >= {threshold} to {len(df_exp)} articles...")

    crossref_fields = (
        "doi", "crossref_title", "crossref_authors", "crossref_year",
        "crossref_references", "crossref_subject", "crossref_journal",
        "crossref_publisher", "crossref_type", "crossref_abstract",
        "crossref_issn", "crossref_volume", "crossref_issue",
        "crossref_page", "crossref_event_name", "crossref_language",
    )

    enriched_rows = []
    for _, row in df_exp.iterrows():
        base = {
            "pdf_file":         row["pdf_file"],
            "original_title":   row["original_title"],
            "original_authors": row["original_authors"],
            "original_year":    row["original_year"],
        }

        accepted = False
        if row["search_strategy"] == "with_title":
            sim = row.get("similarity_1")
            accepted = pd.notna(sim) and sim >= threshold
            if accepted:
                base["title_similarity"] = sim
        elif row["search_strategy"] == "author_only":
            accepted = pd.notna(row.get("doi_1"))

        if accepted:
            for field in crossref_fields:
                base[field] = row.get(f"{field}_1", "")

        base["enriched"] = accepted
        enriched_rows.append(base)

    df_final = pd.DataFrame(enriched_rows)
    df_final.to_csv(output_csv, index=False, encoding="utf-8")

    total_enriched = df_final["enriched"].sum()
    print(f"\nSaved {len(df_final)} rows -> {output_csv}")
    print(f"  Enriched: {total_enriched} ({total_enriched / len(df_final) * 100:.1f}%)")
    print(f"  Not enriched: {(~df_final['enriched']).sum()}")
    return df_final

In [ ]:
df_experiment = enrich_dataset_with_crossref(
    input_csv=INPUT_CSV,
    output_csv=EXPERIMENT_CSV,
)

df_final = apply_threshold_and_enrich(
    experiment_csv=EXPERIMENT_CSV,
    output_csv=ENRICHED_CSV,
    threshold=SIMILARITY_THRESHOLD,
)

print(f"\nEnriched: {df_final['enriched'].sum()} / {len(df_final)} articles")
print(f"Final dataset saved to: {ENRICHED_CSV}")

## Validation (optional)

Generates a simplified CSV sorted by similarity score for manual spot-checking of match quality.

In [ ]:
df_check = pd.read_csv(EXPERIMENT_CSV)

df_check = df_check[
    ["idx", "pdf_file", "original_title", "original_authors", "original_year",
     "crossref_title_1", "crossref_authors_1", "crossref_year_1", "similarity_1", "doi_1"]
].copy()

df_check["is_correct"] = ""  # column for manual annotation
df_check = df_check.sort_values("similarity_1", ascending=False, na_position="last")
df_check.to_csv(CHECK_CSV, index=False, encoding="utf-8")

sim = df_check["similarity_1"]
print(f"similarity >= 0.9:    {(sim >= 0.9).sum()} articles")
print(f"similarity 0.8–0.9:  {((sim >= 0.8) & (sim < 0.9)).sum()} articles")
print(f"similarity 0.7–0.8:  {((sim >= 0.7) & (sim < 0.8)).sum()} articles")
print(f"similarity < 0.7:    {(sim < 0.7).sum()} articles")
print(f"\nValidation file saved to: {CHECK_CSV}")